In [10]:
"""
聪明钱因子 - 数据导出脚本（聚宽研究环境）
导出内容：
  1. min30_YYYY-MM-DD.pkl    : 每月末前10天30分钟数据 + 已过滤股票池
  2. rebalance_close.parquet : 所有月初调仓日的收盘价
  3. schedule.csv            : 调仓日程表（月末算因子 → 月初调仓）
"""

import os
import pickle
import pandas as pd
from datetime import datetime
from jqdata import *  # 导入聚宽数据模块

# ==================== 配置 ====================
SAVE_DIR = '/tmp/smart_money_export'   # 聚宽内保存路径
os.makedirs(SAVE_DIR, exist_ok=True)

START_DATE = '2013-04-01'   # 首次因子计算日（月末）
END_DATE   = '2019-12-31'   # 末次因子计算日
LOOKBACK   = 10             # 回溯10个交易日
FREQ       = '30m'          # 30分钟K线

# 1. 股票池获取（含过滤）

def get_stockpool(symbol, watch_date):
    """
    获取已过滤的股票池：
      - 去重
      - 过滤ST
      - 过滤上市不足60日新股
      - 过滤当日停牌（我这里没有过滤这个条件）
    """
    # 取指数成分股
    if symbol == 'A':
        stocks = get_index_stocks('000001.XSHG', date=watch_date) + \
                 get_index_stocks('399107.XSHE', date=watch_date)
    else:
        stocks = get_index_stocks(symbol, date=watch_date)
    
    stocks = sorted(list(set(stocks)))
    
    # 过滤ST：is_st=True 表示是ST股，保留 False 的
    st_info = get_extras('is_st', stocks, end_date=watch_date, df=True, count=1).iloc[0]
    stocks = st_info[st_info == False].dropna().index.tolist()
    
    # 过滤上市不足60日：用 get_all_securities 批量取上市日，比逐个 get_security_info 快
    sec_info = get_all_securities(date=watch_date)
    sec_info['days'] = (pd.to_datetime(watch_date) - pd.to_datetime(sec_info['start_date'])).dt.days
    valid = sec_info[sec_info['days'] > 60].index.tolist()
    stocks = [s for s in stocks if s in valid]
    
#     # 过滤当日停牌：paused=1 表示停牌，保留 paused=0 的
#     paused = get_price(stocks, end_date=watch_date, count=1, fields='paused', panel=False)
#     stocks = paused.query('paused != 1')['code'].values.tolist()
    
    return stocks

#  2. 生成调仓日程表

# 获取区间内所有交易日
all_days = get_trade_days(start_date='2013-04-01', end_date='2020-02-28')
print("最早交易日:", all_days[0])  # 应该是 '2013-04-01'
print("最早月末:", all_days[-1] if len(all_days) > 0 else "无数据")

# 每月最后一个交易日（因子计算日）
df_days = pd.DataFrame({'date': all_days})
df_days['date'] = pd.to_datetime(df_days['date']) 
df_days['ym'] = df_days['date'].dt.to_period('M')
calc_dates = df_days.groupby('ym')['date'].max().tolist()          # 月末
calc_dates = [d for d in calc_dates if START_DATE <= d.strftime('%Y-%m-%d') <= END_DATE]

# 每月第一个交易日（实际调仓日）
rebalance_dates = df_days.groupby('ym')['date'].min().tolist()     # 月初

# 对齐：月末算因子 → 次月初调仓。去掉第一个月初和最后一个月末
schedule = pd.DataFrame({
    'calc_date': [d.strftime('%Y-%m-%d') for d in calc_dates],
    'rebalance_date': [d.strftime('%Y-%m-%d') for d in rebalance_dates[1:1+len(calc_dates)]]
})
schedule.to_csv(f'{SAVE_DIR}/schedule.csv', index=False)
print(f"日程表已生成：{len(schedule)} 个月 | 首条: {schedule.iloc[0].to_dict()}")

# 3. 批量导出30分钟数据

print(f"\n开始导出30分钟数据（共{len(calc_dates)}个月，约需20-40分钟）...")

for i, calc_date in enumerate(calc_dates):
    date_str = calc_date.strftime('%Y-%m-%d')
    filepath = f'{SAVE_DIR}/min30_{date_str}.pkl'
    
    # 断点续传：已存在则跳过
    if os.path.exists(filepath):
        print(f"[{i+1}/{len(calc_dates)}] {date_str} 已存在，跳过")
        continue
    
    # 获取当月过滤后的股票池
    stock_pool = get_stockpool('A', date_str)
    
    # 获取30分钟数据：10天约80根30m线，取120根保险
    df = get_price(stock_pool, end_date=date_str, frequency=FREQ,
                   fields=['open','close','high','low','volume','money'],
                   count=120, panel=False, skip_paused=False)
    
    # 清洗：只保留最近10个交易日
    df['trade_date'] = df['time'].dt.date
    valid_days = sorted(df['trade_date'].unique())[-LOOKBACK:]
    df = df[df['trade_date'].isin(valid_days)]
    
    # 打包保存：股票池 + 分钟数据（股票池已过滤，本地无需再过滤）
    package = {'stock_pool': stock_pool, 'data': df}
    with open(filepath, 'wb') as f:
        pickle.dump(package, f)
    
    print(f"[{i+1}/{len(calc_dates)}] {date_str} | 股票{len(stock_pool)}只 | 已保存")

# ==================== 4. 导出调仓日close ====================

print(f"\n开始导出调仓日close...")

# 收集所有出现过的股票
all_stocks = set()
for calc_date in calc_dates:
    with open(f'{SAVE_DIR}/min30_{calc_date.strftime("%Y-%m-%d")}.pkl', 'rb') as f:
        all_stocks.update(pickle.load(f)['stock_pool'])
all_stocks = list(all_stocks)

# 所有需要close的日期：每个调仓日 + 最后一次平仓日
need_dates = schedule['rebalance_date'].unique().tolist()
last_rebalance = pd.Timestamp(need_dates[-1])
future_rebalance = [d for d in rebalance_dates if d > last_rebalance]
last_exit = future_rebalance[0] if future_rebalance else last_rebalance
need_dates.append(last_exit.strftime('%Y-%m-%d'))

# 逐日获取close
records = []
for date in need_dates:
    df = get_price(all_stocks, end_date=date, count=1, fields='close', panel=False)
    df['date'] = date
    records.append(df[['date', 'code', 'close']])

close_df = pd.concat(records, ignore_index=True)
close_df.to_pickle(f'{SAVE_DIR}/rebalance_close.pkl')

print(f"调仓日close已保存：{len(need_dates)} 个日期 | {len(close_df)} 条记录")
print(f"\n全部完成！请从左侧文件树进入 {SAVE_DIR}，右键下载到本地。")

最早交易日: 2013-04-01
最早月末: 2020-02-28
日程表已生成：81 个月 | 首条: {'calc_date': '2013-04-26', 'rebalance_date': '2013-05-02'}

开始导出30分钟数据（共81个月，约需20-40分钟）...
[1/81] 2013-04-26 已存在，跳过
[2/81] 2013-05-31 已存在，跳过
[3/81] 2013-06-28 已存在，跳过
[4/81] 2013-07-31 已存在，跳过
[5/81] 2013-08-30 已存在，跳过
[6/81] 2013-09-30 已存在，跳过
[7/81] 2013-10-31 已存在，跳过
[8/81] 2013-11-29 已存在，跳过
[9/81] 2013-12-31 已存在，跳过
[10/81] 2014-01-30 已存在，跳过
[11/81] 2014-02-28 已存在，跳过
[12/81] 2014-03-31 已存在，跳过
[13/81] 2014-04-30 已存在，跳过
[14/81] 2014-05-30 已存在，跳过
[15/81] 2014-06-30 已存在，跳过
[16/81] 2014-07-31 已存在，跳过
[17/81] 2014-08-29 已存在，跳过
[18/81] 2014-09-30 已存在，跳过
[19/81] 2014-10-31 已存在，跳过
[20/81] 2014-11-28 已存在，跳过
[21/81] 2014-12-31 已存在，跳过
[22/81] 2015-01-30 已存在，跳过
[23/81] 2015-02-27 已存在，跳过
[24/81] 2015-03-31 已存在，跳过
[25/81] 2015-04-30 已存在，跳过
[26/81] 2015-05-29 已存在，跳过
[27/81] 2015-06-30 已存在，跳过
[28/81] 2015-07-31 已存在，跳过
[29/81] 2015-08-31 已存在，跳过
[30/81] 2015-09-30 已存在，跳过
[31/81] 2015-10-30 已存在，跳过
[32/81] 2015-11-30 已存在，跳过
[33/81] 2015-12-31 已存在，跳过
[34/8

In [13]:
import shutil
import os

src = '/tmp/smart_money_export'
dst_zip = './smart_money_export.zip'

# 打包
shutil.make_archive(dst_zip.replace('.zip', ''), 'zip', src)

# 显示大小
size_mb = os.path.getsize(dst_zip) / 1024 / 1024
print(f"打包完成: {dst_zip}")
print(f"文件大小: {size_mb:.1f} MB")

打包完成: ./smart_money_export.zip
文件大小: 437.8 MB
请在左侧文件树找到 smart_money_export.zip，右键 → 下载
